# Session 5: Working with Libraries

**Course:** Python for Data Engineering  
**Phase 2:** Data Handling & Transformation

**What we'll cover:**
- Package management with pip
- Virtual environments
- Common data engineering libraries overview
- Installing and verifying pandas, numpy

**Note:** This session is mostly about setup and understanding the ecosystem. Less coding, more tooling.

---

## 1. Why Libraries Matter

In Phase 1, we did everything with Python's built-in modules (`csv`, `json`, `logging`). That works, but real pipelines use third-party libraries that are faster and more powerful.

For example:
- **pandas** — read a CSV and do in 2 lines what took us 20 lines with `csv.DictReader`
- **numpy** — fast math on large arrays
- **requests** — call APIs easily
- **sqlalchemy** — connect to any database with the same code

But before you install anything, you need to understand **where** packages go and how to manage them.

---

## 2. pip — Python's Package Manager

`pip` installs packages from [PyPI](https://pypi.org) (Python Package Index) — a public repository with 400,000+ packages.

### Common commands

| Command | What it does |
|---------|-------------|
| `pip install pandas` | Install a package |
| `pip install pandas==2.1.0` | Install a specific version |
| `pip install --upgrade pandas` | Upgrade to latest |
| `pip uninstall pandas` | Remove a package |
| `pip list` | Show all installed packages |
| `pip freeze` | Show installed packages in requirements format |
| `pip show pandas` | Show details about a package |

In [ ]:
# Check what's already installed
# The ! prefix runs shell commands from a notebook

!pip list --format=columns

In [ ]:
# pip freeze — this format is what you put in requirements.txt

!pip freeze

---

## 3. Virtual Environments

**Problem:** If you install packages globally, different projects can conflict. Project A needs `pandas 1.5`, Project B needs `pandas 2.1` — one breaks.

**Solution:** Virtual environments. Each project gets its own isolated Python with its own packages.

### Creating and using a venv

```bash
# Create a virtual environment
python -m venv .venv

# Activate it
# Windows:
.venv\Scripts\activate
# Mac/Linux:
source .venv/bin/activate

# Now pip install goes into THIS environment only
pip install pandas numpy

# Deactivate when done
deactivate
```

### The workflow in real projects

```bash
# 1. Create venv (once per project)
python -m venv .venv

# 2. Activate
source .venv/bin/activate    # or .venv\Scripts\activate on Windows

# 3. Install dependencies
pip install pandas numpy requests

# 4. Save them so others can reproduce
pip freeze > requirements.txt

# 5. Someone else clones your project and runs:
pip install -r requirements.txt
```

**Always add `.venv/` to `.gitignore`** — you don't commit the environment, just the `requirements.txt`.

In [ ]:
# Check which Python/pip you're using — this tells you if you're in a venv

import sys
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")

# If you see '.venv' in the path, you're in a virtual environment
in_venv = hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
print(f"In virtual env: {in_venv}")

### requirements.txt

This is the standard way to declare project dependencies. Every data engineering project should have one.

```
pandas==2.1.0
numpy==1.24.3
requests==2.31.0
sqlalchemy==2.0.20
```

**Why pin versions?** Without them, `pip install pandas` grabs the latest — which might break your code if a new version changes behavior. In production, always pin.

**Try it:** Run `!pip freeze` in the cell above, then check:
1. Is pandas installed? What version?
2. Is numpy installed? What version?
3. If not, install them: `!pip install pandas numpy`

In [ ]:
# Your code here

---

## 4. Common Data Engineering Libraries

Here's what you'll encounter in real DE work. We'll use many of these in later sessions.

### Data Processing

| Library | What it does | When to use |
|---------|-------------|------------|
| **pandas** | DataFrames, data cleaning, CSV/JSON/Excel I/O | Most data work, small-medium data |
| **numpy** | Fast math on arrays | Numeric transformations, aggregations |
| **polars** | Faster alternative to pandas | Large datasets, performance-critical |

### APIs & Web

| Library | What it does | When to use |
|---------|-------------|------------|
| **requests** | HTTP calls | Fetching data from REST APIs |
| **beautifulsoup4** | HTML parsing | Web scraping |

### Databases

| Library | What it does | When to use |
|---------|-------------|------------|
| **sqlalchemy** | SQL toolkit, ORM | Connecting to any SQL database |
| **psycopg2** | PostgreSQL driver | Direct Postgres connections |
| **sqlite3** | SQLite (built-in) | Local databases, testing |

### Big Data

| Library | What it does | When to use |
|---------|-------------|------------|
| **pyspark** | Apache Spark Python API | Distributed processing, TB-scale |
| **dask** | Parallel pandas | Data too big for pandas but not Spark-level |

---

## 5. Installing and Verifying Libraries

Let's install the core libraries we'll use in Phase 2 and verify they work.

In [ ]:
# Install pandas and numpy (skip if already installed)

!pip install pandas numpy

In [ ]:
# Verify installation

import pandas as pd
import numpy as np

print(f"pandas version: {pd.__version__}")
print(f"numpy version:  {np.__version__}")

---

## 6. Quick Preview — Why pandas Changes Everything

Remember how we read and cleaned `sales.csv` in Sessions 2 and 3? Here's the same thing with pandas.

In [ ]:
# How we did it before — manual CSV processing
# (from Session 3)

import csv

clean_sales = []
with open("data/sales.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        quantity = int(row["quantity"])
        if quantity <= 0:
            continue
        price = float(row["unit_price"].replace("$", ""))
        clean_sales.append({
            "product": row["product"].strip().title(),
            "quantity": quantity,
            "unit_price": price,
            "total": round(quantity * price, 2),
        })

for sale in clean_sales:
    print(f"  {sale['product']:12s} qty={sale['quantity']} total=${sale['total']:>10,.2f}")

In [ ]:
# How you do it with pandas — same result, far less code

import pandas as pd

df = pd.read_csv("data/sales.csv")
df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False).astype(float)
df["total"] = df["quantity"] * df["unit_price"]
df = df[df["quantity"] > 0]

print(df.to_string(index=False))

That's 4 lines vs 15. And pandas handles edge cases, dtypes, and performance for you. We'll dive deep into pandas in Session 6.

---

## 7. numpy Basics — What You Need for DE

numpy is the foundation under pandas. You won't use it directly that often, but understanding arrays helps you understand DataFrames.

In [ ]:
import numpy as np

# numpy arrays vs Python lists — speed matters at scale

prices = [999.99, 29.99, 79.50, 349.99, 59.99]
quantities = [2, 10, 5, 1, 3]

# With plain Python — loop through each pair
totals_python = [p * q for p, q in zip(prices, quantities)]
print(f"Python list: {totals_python}")

# With numpy — operates on the whole array at once (vectorized)
np_prices = np.array(prices)
np_quantities = np.array(quantities)
totals_numpy = np_prices * np_quantities
print(f"numpy array: {totals_numpy}")

# numpy aggregations
print(f"\nTotal revenue: ${totals_numpy.sum():,.2f}")
print(f"Average sale:  ${totals_numpy.mean():,.2f}")
print(f"Max sale:      ${totals_numpy.max():,.2f}")

The key idea: numpy does math on **entire arrays at once** instead of looping. This is called **vectorization** and it's 10-100x faster than loops for large datasets.

---

## Lab Exercises

---

### Lab 1: Set Up Your Project

Create a proper `requirements.txt` for this project.

**Steps:**
1. Run `!pip freeze` and identify the packages we're using
2. Create a `requirements.txt` file with pinned versions for: `pandas`, `numpy`
3. Verify by reading the file back

You can write files from a notebook using `%%writefile`:
```python
%%writefile requirements.txt
pandas==2.1.0
numpy==1.24.3
```

In [ ]:
# Your code here

---

### Lab 2: Before & After Comparison

Read `data/employees.csv` two ways and compare:

1. **Manual way** (Session 3 style): `csv.DictReader`, loop, clean, convert types
2. **pandas way**: `pd.read_csv()`, use DataFrame methods

For both approaches:
- Read the file
- Show the first 3 records
- Calculate total salary
- Find the highest-paid employee

In [ ]:
import csv
import pandas as pd

# Your code here

---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| pip | `pip install`, `pip freeze`, `pip show` — manage packages |
| venv | `python -m venv .venv` — isolate project dependencies |
| requirements.txt | Pin versions, share with team, reproduce environments |
| pandas | DataFrames for tabular data — replaces manual CSV work |
| numpy | Fast array math — vectorization over loops |

**Key patterns:**
- Always use a virtual environment per project
- Pin dependency versions in `requirements.txt`
- pandas replaces most of the manual file + loop work from Phase 1
- numpy gives you vectorized operations — no loops for math

**Next session:** Pandas Fundamentals — DataFrames, Series, reading data, and exploring datasets.